# 07 — Threads, Locks, Queues, and Shared State

Threads are useful for blocking I/O, but concurrent access to mutable state needs synchronization. Learn locks, queues, bounded pools, and the limits of the GIL.

**Level:** Advanced · **Study time:** 90–150 minutes · **Environment:** Python 3.11+ (validated with 3.12); standard library only.

## Learning objectives
- Submit tasks using ThreadPoolExecutor
- Use Lock for shared mutable state
- Communicate via queue.Queue
- Avoid assuming the GIL makes all compound operations safe

**Method:** Read, predict, execute, vary inputs, and test error paths. The examples avoid live network requests and lengthy sleeps. If an environment has already started an event loop, prefer the standalone asyncio.run examples in a fresh kernel where appropriate.

## Guided lessons

Each section includes theory, runnable code, and a reasoning prompt. Advanced code should minimize hidden state and make cancellation, ownership, and failure behavior explicit.

### 1. Thread pools and results

A ThreadPoolExecutor manages a bounded set of worker threads. Use a with context so the pool shuts down and completes submitted work. Map preserves input order; futures offer completion-based APIs.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def square(n):
    return n * n

with ThreadPoolExecutor(max_workers=3) as pool:
    values = list(pool.map(square, [1, 2, 3, 4]))
print(values)

**Think:** Why can the ordering of individual completion times differ from map output order?

### 2. Protecting shared state

A lock makes critical sections mutually exclusive. Even if individual operations are atomic on a specific interpreter, read-modify-write sequences should not rely on that. Prefer local state and message passing when practical.

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor

counter = {"value": 0}
lock = threading.Lock()

def increment_many(times):
    for _ in range(times):
        with lock:
            counter["value"] += 1

with ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(increment_many, [100] * 4))
print(counter)
assert counter["value"] == 400

**Think:** What race might occur between reading, adding one, and writing without synchronization?

### 3. Queues communicate work

queue.Queue is a thread-safe producer–consumer channel. A bounded maxsize creates backpressure. Each submitted item should receive a corresponding task_done call if consumers use queue.join.

In [ ]:
from queue import Queue
import threading

q = Queue(maxsize=3)
results = []

def consume():
    while True:
        item = q.get()
        try:
            if item is None:
                return
            results.append(item * 2)
        finally:
            q.task_done()

thread = threading.Thread(target=consume)
thread.start()
for value in (1, 2, 3):
    q.put(value)
q.put(None)
q.join()
thread.join()
print(results)

**Think:** Why is a bounded queue valuable during overload?

### 4. Thread safety of APIs

Threading introduces nondeterministic scheduling, so tests should assert invariants rather than an arbitrary execution order. Protect shared data, avoid holding locks while doing slow I/O, and establish consistent lock ordering to prevent deadlocks.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def label(n):
    return f"task-{n}"

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(label, n) for n in range(4)]
    labels = {future.result() for future in as_completed(futures)}
print(sorted(labels))
assert len(labels) == 4

**Think:** What changes in tests when result ordering is not guaranteed?

## Exercises

Build a small implementation and test at least one normal and one invalid case. Document timeouts, cancellation rules, or state ownership when relevant.

**Exercise 1.** Implement a counter with and without a lock and explain nondeterministic outcomes.

In [ ]:
# Exercise 1
# TODO: Write code and tests here.

**Exercise 2.** Build a producer–consumer pipeline with a bounded Queue and sentinel shutdown.

In [ ]:
# Exercise 2
# TODO: Write code and tests here.

**Exercise 3.** Use as_completed to collect results while preserving correct task identification.

In [ ]:
# Exercise 3
# TODO: Write code and tests here.

**Exercise 4.** Explain when asyncio, threads, or processes are appropriate.

In [ ]:
# Exercise 4
# TODO: Write code and tests here.

## Pitfalls and reflection

- The GIL does not guarantee correctness of compound shared-state operations.
- Use context managers for executors and locks.
- Do not rely on task completion order in concurrent tests.

**Checkpoint:** Can you explain the failure path and the resource lifetime? What assumption would fail at larger scale?

Restart the kernel and run all cells before moving to the next lesson.